# Raw video VideoMAE RunPod experiment - 100 per label

?? 10? ?? ??? ??? ?? VideoMAE ??? ????.

?? ??:
- ?? ?? ??? 100?
- ? 400? ?? ??
- train/val/test = 70/20/10
- frame_count=16

??:
1. freeze_backbone=True, lr=1e-4
2. freeze_backbone=False, lr=1e-5

??:
- ? ???? RunPod ?? `/workspace/SKN27-FINAL-3Team` ??? ????.


In [1]:
from pathlib import Path
import csv
import json
import os
import subprocess
import sys
from collections import Counter, defaultdict

RUN_INSTALL_REQUIREMENTS = True
RUN_DOWNLOAD = True
RUN_TRAIN_EXP1 = True
RUN_TRAIN_EXP2 = True
RUN_TRAIN_EXP3 = True

PROJECT_ROOT = Path('/workspace/SKN27-FINAL-3Team')
if not PROJECT_ROOT.exists():
    raise FileNotFoundError(f'RunPod project root not found: {PROJECT_ROOT}')

MANIFEST_DIR = PROJECT_ROOT / 'storage/vision/datasets/classification/manifests'
RAW_VIDEO_DIR = PROJECT_ROOT / 'storage/vision/datasets/classification/raw_videos'
MODEL_DIR = PROJECT_ROOT / 'storage/vision/models/videomae_raw_video'
SAMPLE_MANIFEST = MANIFEST_DIR / 'sample_700_coarse_manifest.csv'
FULL_DOWNLOAD_MANIFEST = MANIFEST_DIR / 'train_700_download_manifest.csv'
MANIFEST_100 = MANIFEST_DIR / 'train_100_raw_video_manifest.csv'
SPLIT_MANIFEST_100 = MANIFEST_DIR / 'train_100_raw_video_manifest_split.csv'

FRAME_COUNT = 16
BATCH_SIZE = 1
EPOCHS = 5
SEED = 42
DEVICE = 'auto'

print('PROJECT_ROOT:', PROJECT_ROOT)
print('FULL_DOWNLOAD_MANIFEST:', FULL_DOWNLOAD_MANIFEST)
print('SPLIT_MANIFEST_100:', SPLIT_MANIFEST_100)


PROJECT_ROOT: /workspace/SKN27-FINAL-3Team
FULL_DOWNLOAD_MANIFEST: /workspace/SKN27-FINAL-3Team/storage/vision/datasets/classification/manifests/train_700_download_manifest.csv
SPLIT_MANIFEST_100: /workspace/SKN27-FINAL-3Team/storage/vision/datasets/classification/manifests/train_100_raw_video_manifest_split.csv


In [2]:
def run_command(command, *, enabled=True, timeout=None):
    command = list(map(str, command))
    print('$', ' '.join(command), flush=True)
    if not enabled:
        print('SKIPPED')
        return None
    env = os.environ.copy()
    env['PYTHONIOENCODING'] = 'utf-8'
    process = subprocess.Popen(
        command,
        cwd=PROJECT_ROOT,
        text=True,
        encoding='utf-8',
        errors='replace',
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        env=env,
    )
    try:
        for line in process.stdout:
            print(line, end='')
        returncode = process.wait(timeout=timeout)
    except Exception:
        process.kill()
        raise
    if returncode != 0:
        raise subprocess.CalledProcessError(returncode, command)
    return returncode


def read_csv(path):
    with Path(path).open('r', encoding='utf-8', newline='') as f:
        return list(csv.DictReader(f))


def write_csv(rows, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    fields = list(dict.fromkeys(key for row in rows for key in row.keys()))
    with path.open('w', encoding='utf-8', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=fields)
        writer.writeheader()
        writer.writerows(rows)


def resolve_local_video_path(path_value):
    if not path_value:
        return None
    path_text = str(path_value).replace('\\', '/')
    runpod_prefix = '/workspace/SKN27-FINAL-3Team/'
    if path_text.startswith(runpod_prefix):
        return PROJECT_ROOT / path_text[len(runpod_prefix):]
    path = Path(path_value)
    return path if path.is_absolute() else PROJECT_ROOT / path


def make_subset_manifest(source_manifest, output_manifest, per_label):
    rows = read_csv(source_manifest)
    selected = []
    counts = defaultdict(int)
    for row in rows:
        label = row.get('coarse_label') or row.get('label')
        if not label or counts[label] >= per_label:
            continue
        path = resolve_local_video_path(row.get('local_path') or row.get('file_path'))
        if path is not None:
            if not path.exists():
                continue
            row = dict(row)
            row['local_path'] = str(path)
            row['file_exists'] = 'True'
        selected.append(row)
        counts[label] += 1
    write_csv(selected, output_manifest)
    print('subset_manifest:', output_manifest)
    print('rows:', len(selected))
    print('label_counts:', dict(Counter(row.get('coarse_label') for row in selected)))
    return output_manifest


def make_split_manifest(source_manifest, output_manifest, train_ratio=0.7, val_ratio=0.2):
    rows = read_csv(source_manifest)
    grouped = defaultdict(list)
    for row in rows:
        grouped[row.get('coarse_label')].append(row)
    split_rows = []
    for label, label_rows in grouped.items():
        total = len(label_rows)
        train_end = int(total * train_ratio)
        val_end = train_end + int(total * val_ratio)
        for index, row in enumerate(label_rows):
            copied = dict(row)
            if index < train_end:
                copied['split'] = 'train'
            elif index < val_end:
                copied['split'] = 'val'
            else:
                copied['split'] = 'test'
            split_rows.append(copied)
    write_csv(split_rows, output_manifest)
    print('split_manifest:', output_manifest)
    print('rows:', len(split_rows))
    print('label_counts:', dict(Counter(row.get('coarse_label') for row in split_rows)))
    print('split_counts:', dict(Counter(row.get('split') for row in split_rows)))
    print('label_split_counts:')
    for key, value in sorted(Counter((row.get('coarse_label'), row.get('split')) for row in split_rows).items()):
        print(key, value)
    return output_manifest


def latest_run_dir(path):
    runs = [p for p in Path(path).glob('videomae_cls_*') if p.is_dir()]
    if not runs:
        raise FileNotFoundError(f'No runs found under {path}')
    return sorted(runs)[-1]


def build_train_command(experiment):
    command = [
        sys.executable,
        'ai/vision/train_videomae_classifier.py',
        '--manifest', experiment['manifest'],
        '--root-dir', PROJECT_ROOT,
        '--output-dir', experiment['output_dir'],
        '--label-column', 'coarse_label',
        '--frame-count', experiment['frame_count'],
        '--epochs', experiment['epochs'],
        '--batch-size', experiment['batch_size'],
        '--learning-rate', experiment['learning_rate'],
        '--weight-decay', experiment['weight_decay'],
        '--early-stopping-patience', experiment['early_stopping_patience'],
        '--seed', SEED,
        '--device', DEVICE,
        '--num-workers', '0',
        '--no-show-progress',
    ]
    if experiment['freeze_backbone']:
        command.append('--freeze-backbone')
    return command


def run_experiment(experiment, *, enabled=True):
    print('\n##', experiment['name'])
    print(json.dumps({k: str(v) for k, v in experiment.items()}, ensure_ascii=False, indent=2))
    run_command(build_train_command(experiment), enabled=enabled, timeout=None)
    if enabled:
        print('LAST_RUN_DIR:', latest_run_dir(experiment['output_dir']))


In [3]:
requirements_file = PROJECT_ROOT / 'requirements-vision-runpod.txt'
if not requirements_file.exists():
    requirements_file = PROJECT_ROOT / 'requirements.txt'
run_command([sys.executable, '-m', 'pip', 'install', '-r', requirements_file], enabled=RUN_INSTALL_REQUIREMENTS, timeout=3600)

if not SAMPLE_MANIFEST.exists() and not FULL_DOWNLOAD_MANIFEST.exists():
    raise FileNotFoundError(f'Missing source manifest: {SAMPLE_MANIFEST}')

if not FULL_DOWNLOAD_MANIFEST.exists():
    run_command([
        sys.executable,
        'etl/vision/download_sampled_media.py',
        '--input', SAMPLE_MANIFEST,
        '--output', FULL_DOWNLOAD_MANIFEST,
        '--download-dir', RAW_VIDEO_DIR,
        '--label-column', 'coarse_label',
        '--per-label', '700',
        '--split', '',
    ], enabled=RUN_DOWNLOAD, timeout=None)

rows = read_csv(FULL_DOWNLOAD_MANIFEST)
print('download_rows:', len(rows))
print('label_counts:', dict(Counter(row.get('coarse_label') for row in rows)))
print('download_status:', dict(Counter(row.get('download_status') for row in rows)))
print('file_exists:', dict(Counter(row.get('file_exists') for row in rows)))


$ /usr/bin/python -m pip install -r /workspace/SKN27-FINAL-3Team/requirements-vision-runpod.txt

[notice] A new release of pip is available: 24.2 -> 26.1.2
[notice] To update, run: python -m pip install --upgrade pip
download_rows: 2800
label_counts: {'차대보행자': 700, '차대이륜차': 700, '차대자전거': 700, '차대차': 700}
download_status: {'exists': 1370, 'downloaded': 1430}
file_exists: {'True': 2800}


In [4]:
make_subset_manifest(FULL_DOWNLOAD_MANIFEST, MANIFEST_100, per_label=100)
make_split_manifest(MANIFEST_100, SPLIT_MANIFEST_100)


subset_manifest: /workspace/SKN27-FINAL-3Team/storage/vision/datasets/classification/manifests/train_100_raw_video_manifest.csv
rows: 400
label_counts: {'차대보행자': 100, '차대이륜차': 100, '차대자전거': 100, '차대차': 100}
split_manifest: /workspace/SKN27-FINAL-3Team/storage/vision/datasets/classification/manifests/train_100_raw_video_manifest_split.csv
rows: 400
label_counts: {'차대보행자': 100, '차대이륜차': 100, '차대자전거': 100, '차대차': 100}
split_counts: {'train': 280, 'val': 80, 'test': 40}
label_split_counts:
('차대보행자', 'test') 10
('차대보행자', 'train') 70
('차대보행자', 'val') 20
('차대이륜차', 'test') 10
('차대이륜차', 'train') 70
('차대이륜차', 'val') 20
('차대자전거', 'test') 10
('차대자전거', 'train') 70
('차대자전거', 'val') 20
('차대차', 'test') 10
('차대차', 'train') 70
('차대차', 'val') 20


PosixPath('/workspace/SKN27-FINAL-3Team/storage/vision/datasets/classification/manifests/train_100_raw_video_manifest_split.csv')

In [5]:
EXPERIMENT_1 = {
    'name': 'exp1_raw100_freeze_lr1e-4_fc16',
    'manifest': SPLIT_MANIFEST_100,
    'output_dir': MODEL_DIR / 'per_label_100_exp1_freeze_lr1e-4',
    'epochs': EPOCHS,
    'batch_size': BATCH_SIZE,
    'learning_rate': 0.0001,
    'weight_decay': 0.05,
    'early_stopping_patience': 2,
    'frame_count': FRAME_COUNT,
    'freeze_backbone': True,
}
run_experiment(EXPERIMENT_1, enabled=RUN_TRAIN_EXP1)



## exp1_raw100_freeze_lr1e-4_fc16
{
  "name": "exp1_raw100_freeze_lr1e-4_fc16",
  "manifest": "/workspace/SKN27-FINAL-3Team/storage/vision/datasets/classification/manifests/train_100_raw_video_manifest_split.csv",
  "output_dir": "/workspace/SKN27-FINAL-3Team/storage/vision/models/videomae_raw_video/per_label_100_exp1_freeze_lr1e-4",
  "epochs": "5",
  "batch_size": "1",
  "learning_rate": "0.0001",
  "weight_decay": "0.05",
  "early_stopping_patience": "2",
  "frame_count": "16",
  "freeze_backbone": "True"
}
$ /usr/bin/python ai/vision/train_videomae_classifier.py --manifest /workspace/SKN27-FINAL-3Team/storage/vision/datasets/classification/manifests/train_100_raw_video_manifest_split.csv --root-dir /workspace/SKN27-FINAL-3Team --output-dir /workspace/SKN27-FINAL-3Team/storage/vision/models/videomae_raw_video/per_label_100_exp1_freeze_lr1e-4 --label-column coarse_label --frame-count 16 --epochs 5 --batch-size 1 --learning-rate 0.0001 --weight-decay 0.05 --early-stopping-patience 2 

In [6]:
EXPERIMENT_2 = {
    'name': 'exp2_raw100_unfreeze_lr1e-5_fc16',
    'manifest': SPLIT_MANIFEST_100,
    'output_dir': MODEL_DIR / 'per_label_100_exp2_unfreeze_lr1e-5',
    'epochs': EPOCHS,
    'batch_size': BATCH_SIZE,
    'learning_rate': 0.00001,
    'weight_decay': 0.05,
    'early_stopping_patience': 2,
    'frame_count': FRAME_COUNT,
    'freeze_backbone': False,
}
run_experiment(EXPERIMENT_2, enabled=RUN_TRAIN_EXP2)



## exp2_raw100_unfreeze_lr1e-5_fc16
{
  "name": "exp2_raw100_unfreeze_lr1e-5_fc16",
  "manifest": "/workspace/SKN27-FINAL-3Team/storage/vision/datasets/classification/manifests/train_100_raw_video_manifest_split.csv",
  "output_dir": "/workspace/SKN27-FINAL-3Team/storage/vision/models/videomae_raw_video/per_label_100_exp2_unfreeze_lr1e-5",
  "epochs": "5",
  "batch_size": "1",
  "learning_rate": "1e-05",
  "weight_decay": "0.05",
  "early_stopping_patience": "2",
  "frame_count": "16",
  "freeze_backbone": "False"
}
$ /usr/bin/python ai/vision/train_videomae_classifier.py --manifest /workspace/SKN27-FINAL-3Team/storage/vision/datasets/classification/manifests/train_100_raw_video_manifest_split.csv --root-dir /workspace/SKN27-FINAL-3Team --output-dir /workspace/SKN27-FINAL-3Team/storage/vision/models/videomae_raw_video/per_label_100_exp2_unfreeze_lr1e-5 --label-column coarse_label --frame-count 16 --epochs 5 --batch-size 1 --learning-rate 1e-05 --weight-decay 0.05 --early-stopping-pati

In [8]:
EXPERIMENT_3 = {
    'name': 'exp3_raw100_unfreeze_lr1e-5_fc16_e50',
    'manifest': SPLIT_MANIFEST_100,
    'output_dir': MODEL_DIR / 'per_label_100_exp3_unfreeze_lr1e-5_e50',
    'epochs': 50,
    'batch_size': BATCH_SIZE,
    'learning_rate': 0.00001,
    'weight_decay': 0.05,
    'early_stopping_patience': 2,
    'frame_count': FRAME_COUNT,
    'freeze_backbone': False,
}

run_experiment(EXPERIMENT_3, enabled=RUN_TRAIN_EXP3)


## exp3_raw100_unfreeze_lr1e-5_fc16_e50
{
  "name": "exp3_raw100_unfreeze_lr1e-5_fc16_e50",
  "manifest": "/workspace/SKN27-FINAL-3Team/storage/vision/datasets/classification/manifests/train_100_raw_video_manifest_split.csv",
  "output_dir": "/workspace/SKN27-FINAL-3Team/storage/vision/models/videomae_raw_video/per_label_100_exp3_unfreeze_lr1e-5_e50",
  "epochs": "50",
  "batch_size": "1",
  "learning_rate": "1e-05",
  "weight_decay": "0.05",
  "early_stopping_patience": "2",
  "frame_count": "16",
  "freeze_backbone": "False"
}
$ /usr/bin/python ai/vision/train_videomae_classifier.py --manifest /workspace/SKN27-FINAL-3Team/storage/vision/datasets/classification/manifests/train_100_raw_video_manifest_split.csv --root-dir /workspace/SKN27-FINAL-3Team --output-dir /workspace/SKN27-FINAL-3Team/storage/vision/models/videomae_raw_video/per_label_100_exp3_unfreeze_lr1e-5_e50 --label-column coarse_label --frame-count 16 --epochs 50 --batch-size 1 --learning-rate 1e-05 --weight-decay 0.05 --e

In [9]:
def print_latest_history(output_dir):
    run_dir = latest_run_dir(output_dir)
    print('run_dir:', run_dir)
    for file_name in ['run_config.json', 'training_history.csv']:
        path = run_dir / file_name
        print('##', file_name, path.exists())
        if path.suffix == '.json' and path.exists():
            print(json.dumps(json.loads(path.read_text(encoding='utf-8')), ensure_ascii=False, indent=2)[:3000])
        elif path.exists():
            for row in read_csv(path):
                print(row)

if RUN_TRAIN_EXP1:
    print('\n## EXPERIMENT_1 result')
    print_latest_history(EXPERIMENT_1['output_dir'])
if RUN_TRAIN_EXP2:
    print('\n## EXPERIMENT_2 result')
    print_latest_history(EXPERIMENT_2['output_dir'])
if RUN_TRAIN_EXP3:
    print('## EXPERIMENT_3 result')
    print_latest_history(EXPERIMENT_3['output_dir'])


## EXPERIMENT_1 result
run_dir: /workspace/SKN27-FINAL-3Team/storage/vision/models/videomae_raw_video/per_label_100_exp1_freeze_lr1e-4/videomae_cls_20260713_022139
## run_config.json True
{
  "run_id": "videomae_cls_20260713_022139",
  "manifest": "/workspace/SKN27-FINAL-3Team/storage/vision/datasets/classification/manifests/train_100_raw_video_manifest_split.csv",
  "label_column": "coarse_label",
  "model_name": "MCG-NJU/videomae-base-finetuned-kinetics",
  "freeze_backbone": true,
  "frame_count": 16,
  "epochs": 5,
  "batch_size": 1,
  "learning_rate": 0.0001,
  "weight_decay": 0.05,
  "early_stopping_patience": 2,
  "best_epoch": 4,
  "best_val_accuracy": 0.55,
  "seed": 42,
  "device": "cuda",
  "train_rows": 280,
  "val_rows": 80,
  "test_rows": 40,
  "model_path": "/workspace/SKN27-FINAL-3Team/storage/vision/models/videomae_raw_video/per_label_100_exp1_freeze_lr1e-4/videomae_cls_20260713_022139"
}
## training_history.csv True
{'epoch': '1', 'train_loss': '1.372621', 'train_acc

# Optional Qwen2.5-VL raw video scene analysis

VideoMAE 학습과 별개로, split manifest에서 라벨별 샘플 영상을 골라 Qwen2.5-VL 장면 설명 JSON을 생성합니다.
전체 데이터에 Qwen을 돌리면 비용과 시간이 크므로 기본값은 라벨별 1개입니다.


In [ ]:
RUN_QWEN = True
QWEN_SAMPLE_PER_LABEL = 1
QWEN_FPS = 6.4
QWEN_MODEL_ID = "Qwen/Qwen2.5-VL-3B-Instruct"
QWEN_SOURCE_MANIFEST = SPLIT_MANIFEST_100
QWEN_OUTPUT_DIR = PROJECT_ROOT / "storage/vision/outputs/qwen_vl_raw_video_100"

QWEN_PROMPT = 'You are an accident-scene video analysis agent.\nReturn exactly one valid JSON object only. Do not output markdown, code fences, or extra explanation.\nUse the English field names exactly as requested.\nEvery JSON string value MUST be written in Korean language. Do not write English sentences in values.\nDescribe only facts visible in the video.\nDo not decide legal fault ratio, offender, or victim.\nDo not conclude \'no accident\' too strongly. If the accident moment is not clearly visible, write that the accident scene is not clearly identifiable from the sampled video.\n\nRequired fields:\nsummary, visible_objects, accident_situation, scene_conditions, evidence_for_fault_analysis, uncertainties\n\nRequired scene_conditions fields:\nweather, visibility, road_surface, lighting, confidence, evidence\n\nAllowed scene_conditions values:\n- weather: clear | rain | fog | snow | unknown\n- visibility: good | reduced | poor | unknown\n- road_surface: dry | wet | icy | snowy | unknown\n- lighting: day | night | tunnel | backlight | unknown\n- confidence: number from 0.0 to 1.0\n- evidence: Korean evidence text for weather, visibility, road surface, and lighting.\n\nReturn example shape only:\n{"summary":"...","visible_objects":["..."],"accident_situation":"...","scene_conditions":{"weather":"unknown","visibility":"unknown","road_surface":"unknown","lighting":"unknown","confidence":0.0,"evidence":"..."},"evidence_for_fault_analysis":"...","uncertainties":["..."]}\n'


def extract_json_object(text):
    text = text.strip()
    if text.startswith("```"):
        text = text.strip("`").strip()
        if text.lower().startswith("json"):
            text = text[4:].strip()
    start = text.find("{")
    end = text.rfind("}")
    if start == -1 or end == -1 or end <= start:
        raise ValueError("JSON object not found")
    return json.loads(text[start:end + 1])

if RUN_QWEN:
    import torch
    from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration
    from qwen_vl_utils import process_vision_info

    rows = read_csv(QWEN_SOURCE_MANIFEST)
    priority = {"test": 0, "val": 1, "train": 2}
    rows = sorted(rows, key=lambda row: (priority.get(row.get("split"), 9), row.get("coarse_label", ""), row.get("asset_id", "")))

    selected = []
    counts = defaultdict(int)
    for row in rows:
        label = row.get("coarse_label") or row.get("label") or "unknown"
        if counts[label] >= QWEN_SAMPLE_PER_LABEL:
            continue
        video_path = resolve_local_video_path(row.get("local_path") or row.get("file_path"))
        if video_path is None or not video_path.exists():
            continue
        copied = dict(row)
        copied["resolved_video_path"] = str(video_path)
        selected.append(copied)
        counts[label] += 1

    print("qwen_source_manifest:", QWEN_SOURCE_MANIFEST)
    print("selected_rows:", len(selected))
    print("label_counts:", dict(Counter(row.get("coarse_label") for row in selected)))

    model = Qwen2_5_VLForConditionalGeneration.from_pretrained(QWEN_MODEL_ID, torch_dtype="auto", device_map="auto")
    processor = AutoProcessor.from_pretrained(QWEN_MODEL_ID)
    QWEN_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    for row in selected:
        video_path = Path(row["resolved_video_path"])
        asset_id = row.get("asset_id") or video_path.stem
        messages = [{"role": "user", "content": [{"type": "video", "video": str(video_path), "fps": QWEN_FPS}, {"type": "text", "text": QWEN_PROMPT}]}]
        text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        image_inputs, video_inputs = process_vision_info(messages)
        inputs = processor(text=[text], images=image_inputs, videos=video_inputs, padding=True, return_tensors="pt").to(model.device)
        with torch.no_grad():
            generated_ids = model.generate(**inputs, max_new_tokens=512, do_sample=False)
        output_text = processor.batch_decode(generated_ids[:, inputs.input_ids.shape[1]:], skip_special_tokens=True)[0]

        parsed_output = None
        qwen_json_valid = False
        parse_error = None
        try:
            parsed_output = extract_json_object(output_text)
            qwen_json_valid = True
        except Exception as exc:
            parse_error = str(exc)

        out_path = QWEN_OUTPUT_DIR / f"qwen_vl_analysis_{asset_id}.json"
        out_path.write_text(json.dumps({
            "model": QWEN_MODEL_ID,
            "source_manifest": str(QWEN_SOURCE_MANIFEST),
            "row": row,
            "qwen_json_valid": qwen_json_valid,
            "parse_error": parse_error,
            "parsed_output": parsed_output,
            "raw_output_text": output_text,
        }, ensure_ascii=False, indent=2), encoding="utf-8")
        print("
##", row.get("coarse_label"), asset_id)
        print("qwen_json_valid:", qwen_json_valid)
        if parse_error:
            print("parse_error:", parse_error)
        print(output_text)
        print("out_path:", out_path)
else:
    print("RUN_QWEN is False. Set it to True to run Qwen2.5-VL sample analysis.")


In [ ]:
# EXPORT_ANALYSIS_ARTIFACTS
import subprocess
import sys

reports_dir = PROJECT_ROOT / 'storage/vision/reports'
command = [
    sys.executable,
    'ai/vision/export_analysis_artifacts.py',
    '--root-dir', str(PROJECT_ROOT),
    '--output-dir', str(reports_dir),
]
print('$', ' '.join(command))
completed = subprocess.run(command, cwd=PROJECT_ROOT, text=True, capture_output=True, timeout=600)
if completed.stdout:
    print(completed.stdout)
if completed.stderr:
    print(completed.stderr)
completed.check_returncode()
print('tables:', reports_dir / 'tables')
print('figures:', reports_dir / 'figures')
print('appendix:', reports_dir / 'appendix')
